# TraCS run analysis

Aggregates JSON traces in `output/` and reports per-model success rate, average iterations, and the breakdown of syntax vs semantic failures.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(context="paper", style="whitegrid", font_scale=1.2)
sns.set_palette("colorblind")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

OUT = Path("../output")
HUMAN = OUT


assert OUT.exists(), f"Output directory {OUT} does not exist. Please run `make output` to create it."

## Load traces

One row per run (scenario × model × seed).

In [ ]:
rows = []
iter_rows = []
for path in sorted(OUT.glob("*.json")):
    trace = json.loads(path.read_text())
    p = trace["parameters"]
    iters = trace["iterations"]
    rows.append({
        "scenario": p["scenario"],
        "model": p["coder_model"],
        "seed": p["seed"],
        "accepted": trace["accepted"],
        "n_iterations": len(iters),
        "n_syntax_fail": sum(not it["compiled"] for it in iters),
        "n_semantic_fail": sum(not it["judge_passed"] for it in iters),
    })
    for i, it in enumerate(iters, start=1):
        iter_rows.append({
            "scenario": p["scenario"],
            "model": p["coder_model"],
            "seed": p["seed"],
            "iteration": i,
            "compiled": it["compiled"],
            "judge_passed": it["judge_passed"],
        })

df = pd.DataFrame(rows)
iter_df = pd.DataFrame(iter_rows)
print(f"Loaded {len(df)} runs across {df['model'].nunique()} models and {df['scenario'].nunique()} scenarios.")
df.head()

In [ ]:
def heatmap(pivot, ax, *, vmin, vmax, cmap, fmt, cbar_label):
    sns.heatmap(
        pivot, ax=ax, annot=True, fmt=fmt, cmap=cmap, vmin=vmin, vmax=vmax,
        linewidths=0.5, linecolor="white", square=False,
        cbar_kws={"label": cbar_label, "shrink": 0.8},
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=20)
    ax.tick_params(axis="y", rotation=0)

## Success rate

Fraction of seeds that converged, per (model, scenario).

In [ ]:
success_pivot = (
    df.groupby(["model", "scenario"])["accepted"].mean()
    .unstack("scenario")
)
fig, ax = plt.subplots(figsize=(1.0 + 0.9 * success_pivot.shape[1], 0.6 + 0.6 * success_pivot.shape[0]))
heatmap(success_pivot, ax, vmin=0, vmax=1, cmap="Greens", fmt=".2f", cbar_label="Success rate")
plt.show()

In [ ]:
model_success = success_pivot.mean(axis=1).rename("mean_success_rate").reset_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
sns.barplot(data=model_success, x="model", y="mean_success_rate", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean success rate\n(over scenarios)")
ax.set_xlabel("")
plt.show()

## Iterations to convergence

Mean iteration count over seeds for successful runs only. Cells with no successes show as NaN.

In [ ]:
accepted_runs = df[df["accepted"]]
iter_pivot = (
    accepted_runs.groupby(["model", "scenario"])["n_iterations"].mean()
    .unstack("scenario")
)
vmax = max(1, iter_pivot.max().max())
fig, ax = plt.subplots(figsize=(1.0 + 0.9 * iter_pivot.shape[1], 0.6 + 0.6 * iter_pivot.shape[0]))
heatmap(iter_pivot, ax, vmin=1, vmax=vmax, cmap="YlOrRd", fmt=".1f", cbar_label="Mean iterations")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
sns.boxplot(data=accepted_runs, x="model", y="n_iterations", ax=ax, width=0.5)
sns.stripplot(data=accepted_runs, x="model", y="n_iterations", ax=ax,
              color="black", size=3, alpha=0.5, jitter=0.15)
ax.set_ylabel("Iterations to acceptance")
ax.set_xlabel("")
plt.show()

## Failure breakdown: syntax vs semantics

Fraction of iterations that failed compilation (syntax) and that were rejected by the judge (semantics). An iteration can fail on either or both.

In [ ]:
syntax_pivot = (
    iter_df.assign(fail=lambda d: (~d["compiled"]).astype(float))
    .groupby(["model", "scenario"])["fail"].mean()
    .unstack("scenario")
)
semantic_pivot = (
    iter_df.assign(fail=lambda d: (~d["judge_passed"]).astype(float))
    .groupby(["model", "scenario"])["fail"].mean()
    .unstack("scenario")
)

fig, axes = plt.subplots(
    2, 1,
    figsize=(1.0 + 0.9 * syntax_pivot.shape[1], 1.4 + 1.2 * syntax_pivot.shape[0]),
    sharex=True,
)
heatmap(syntax_pivot,   axes[0], vmin=0, vmax=1, cmap="Reds",    fmt=".2f", cbar_label="Syntax fail rate")
heatmap(semantic_pivot, axes[1], vmin=0, vmax=1, cmap="Purples", fmt=".2f", cbar_label="Semantic fail rate")
axes[0].set_title("Syntax")
axes[1].set_title("Semantic")
plt.tight_layout()
plt.show()

In [ ]:
fail_summary = (
    iter_df.assign(
        syntax_fail=lambda d: ~d["compiled"],
        semantic_fail=lambda d: ~d["judge_passed"],
    )
    .groupby("model")[["syntax_fail", "semantic_fail"]]
    .mean()
    .reset_index()
    .melt(id_vars="model", var_name="kind", value_name="fraction")
)
fail_summary["kind"] = fail_summary["kind"].map({"syntax_fail": "Syntax", "semantic_fail": "Semantic"})

fig, ax = plt.subplots(figsize=(5.5, 3.5))
sns.barplot(data=fail_summary, x="model", y="fraction", hue="kind", ax=ax)
ax.set_ylabel("Fraction of iterations failing")
ax.set_xlabel("")
ax.set_ylim(0, 1.05)
ax.legend(title="Failure kind", frameon=False)
plt.show()

## Judge vs human agreement

Some traces have an `opinions` dict per iteration (keyed by judge name). We extract the `"human"` entry and compare its verdict against the LLM pipeline judge's verdict, then produce a confusion matrix.

In [ ]:
human_rows = []
for path in sorted(HUMAN.glob("*.json")):
    trace = json.loads(path.read_text())
    p = trace["parameters"]
    for i, it in enumerate(trace["iterations"], start=1):
        opinions = it.get("opinions", {})
        if "human" not in opinions:
            continue
        human_rows.append({
            "scenario": p["scenario"],
            "model": p["coder_model"],
            "seed": p["seed"],
            "iteration": i,
            "judge_passed": it["judge_passed"],
            "human_passed": opinions["human"]["passed"],
        })

human_df = pd.DataFrame(human_rows)
print(f"Loaded {len(human_df)} iterations with human labels.")
agree = (human_df["judge_passed"] == human_df["human_passed"]).mean() if len(human_df) else float("nan")
print(f"Overall agreement: {agree:.1%}")

In [ ]:
confusion = pd.crosstab(
    human_df["human_passed"].map({True: "accept", False: "reject"}),
    human_df["judge_passed"].map({True: "accept", False: "reject"}),
    rownames=["Human"], colnames=["LLM judge"],
).reindex(index=["accept", "reject"], columns=["accept", "reject"], fill_value=0)
confusion

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.2))
sns.heatmap(
    confusion, ax=ax, annot=True, fmt="d", cmap="Blues",
    linewidths=0.5, linecolor="white", square=True,
    cbar_kws={"label": "Iterations", "shrink": 0.8},
)
ax.set_title(f"Agreement: {agree:.1%}")
plt.show()

In [ ]:
import numpy as np
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar

j = human_df["judge_passed"].astype(int).to_numpy()
h = human_df["human_passed"].astype(int).to_numpy()
n = len(j)

# Cohen's kappa with Fleiss SE for 95% CI and z-test
kappa = cohen_kappa_score(h, j)
p_o = (h == j).mean()
p_h, p_j = h.mean(), j.mean()
p_e = p_h * p_j + (1 - p_h) * (1 - p_j)
se_kappa = np.sqrt(p_o * (1 - p_o) / (n * (1 - p_e) ** 2)) if p_e < 1 else float("nan")
ci_lo, ci_hi = kappa - 1.96 * se_kappa, kappa + 1.96 * se_kappa
z = kappa / se_kappa if se_kappa else float("nan")
p_kappa = 2 * (1 - stats.norm.cdf(abs(z))) if np.isfinite(z) else float("nan")

# Pearson phi (= Pearson on 0/1)
phi, p_phi = stats.pearsonr(h, j)

# Chi-square independence on the 2x2 contingency table
chi2, p_chi2, dof, _ = stats.chi2_contingency(confusion.values)

# McNemar's test for asymmetry of disagreements
mc = mcnemar(confusion.values, exact=True)

def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3g}"

print(f"Cohen's k           : {kappa:.3f}  (95% CI [{ci_lo:.2f}, {ci_hi:.2f}], p {fmt_p(p_kappa)})")
print(f"Pearson phi         : {phi:.3f}  (p {fmt_p(p_phi)})")
print(f"chi^2 independence  : chi2={chi2:.2f}, dof={dof}, p {fmt_p(p_chi2)}")
print(f"McNemar (asymmetry) : p {fmt_p(mc.pvalue)}")

## Model characteristics overview

A radar plot summarizing each model on a [0, 1] scale. Axes, clockwise from the top:

- **Syntax pass rate** — fraction of iterations that compiled.
- **Semantic pass rate** — fraction of iterations the judge passed.
- **1st-attempt success** — fraction of runs accepted on iteration 1 (compiled and judge-passed).
- **Recovery rate** — among runs that failed iteration 1, the fraction the loop eventually rescued. `NaN` if a model never had to recover; the bridging edge is drawn dashed and the axis label gets a `*`.
- **Final success rate** — fraction of runs eventually accepted.
- **Iter. budget left** — `(10 - a) / 9`, where `a` is the iterations an accepted run took. `1.0` means accepted on the first try, `0.0` means the full 10-iteration budget was used. Averaged over accepted runs.
- **Judge–human agreement** — agreement between the LLM judge and the human label, restricted to the model's iterations that have a human label. `NaN` if no labels exist for that model.

The figure is also saved as `output/model_characteristics.pdf` for inclusion in the paper.

In [ ]:
import numpy as np

MAX_ITERATIONS = 10  # see tracs/run_all.py: --max-iterations 10
RECOVERY_LABEL = "Recovery rate"

models = sorted(df["model"].unique())
model_display = {m: m.split("/", 1)[1] if "/" in m else m for m in models}

# --- per-model metrics ---------------------------------------------------

final_success_by_model = df.groupby("model")["accepted"].mean()

# (MAX - a) / (MAX - 1): 1.0 = accepted on first try, 0.0 = full budget used.
budget_left_by_model = (
    df[df["accepted"]]
    .assign(left=lambda d: (MAX_ITERATIONS - d["n_iterations"]) / (MAX_ITERATIONS - 1))
    .groupby("model")["left"].mean()
)

syntax_pass_by_model = (
    iter_df.assign(ok=lambda d: d["compiled"].astype(float))
    .groupby("model")["ok"].mean()
)
semantic_pass_by_model = (
    iter_df.assign(ok=lambda d: d["judge_passed"].astype(float))
    .groupby("model")["ok"].mean()
)

first_attempt_by_model = (
    iter_df[iter_df["iteration"] == 1]
    .assign(ok=lambda d: (d["compiled"] & d["judge_passed"]).astype(float))
    .groupby("model")["ok"].mean()
)

first_iter_pass = (
    iter_df[iter_df["iteration"] == 1]
    .assign(first_ok=lambda d: d["compiled"] & d["judge_passed"])
    .set_index(["model", "scenario", "seed"])["first_ok"]
)
runs = df.set_index(["model", "scenario", "seed"]).join(first_iter_pass)
needed_recovery = runs[~runs["first_ok"]]
recovery_by_model = (
    needed_recovery.groupby(level="model")["accepted"].mean()
    if len(needed_recovery) else pd.Series(dtype=float)
)
recovery_by_model = pd.Series(
    {m: recovery_by_model.get(m, float("nan")) for m in models}
)

if len(human_df):
    alignment_by_model = (
        human_df.assign(agree=lambda d: (
            d["judge_passed"] == d["human_passed"]).astype(float))
        .groupby("model")["agree"].mean()
    )
else:
    alignment_by_model = pd.Series(dtype=float)
alignment_by_model = pd.Series(
    {m: alignment_by_model.get(m, float("nan")) for m in models}
)

# Order: clockwise from top, with shorter labels for paper layout.
metrics = pd.DataFrame({
    "Syntax pass rate": syntax_pass_by_model,
    "Semantic pass rate": semantic_pass_by_model,
    "1st-attempt success": first_attempt_by_model,
    RECOVERY_LABEL: recovery_by_model,
    "Final success rate": final_success_by_model,
    "Iter. budget left": budget_left_by_model,
    "Judge–human agreement": alignment_by_model,
}).reindex(models)

recovery_missing = metrics[RECOVERY_LABEL].isna()
recovery_label_display = RECOVERY_LABEL + \
    (" *" if recovery_missing.any() else "")
display_labels = [
    recovery_label_display if c == RECOVERY_LABEL else c
    for c in metrics.columns
]

# --- plot ---------------------------------------------------------------

n = len(display_labels)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)

fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw={"projection": "polar"})
ax.set_theta_offset(np.pi / 2)   # 0 at the top
ax.set_theta_direction(-1)        # clockwise
palette = sns.color_palette("colorblind", n_colors=len(models))
markers = ["o", "s", "D", "^", "v", "P", "X"]

for color, marker, model in zip(palette, markers, models):
    vals = metrics.loc[model].to_numpy(dtype=float)
    valid = np.where(~np.isnan(vals))[0]
    if len(valid) == 0:
        continue

    # Filled polygon (skips NaN vertices).
    a_fill = np.append(angles[valid], angles[valid[0]])
    v_fill = np.append(vals[valid], vals[valid[0]])
    ax.fill(a_fill, v_fill, color=color, alpha=0.07, zorder=1)

    # Edges: solid between adjacent valid vertices, dashed across NaN bridges.
    for k in range(len(valid)):
        i, j = int(valid[k]), int(valid[(k + 1) % len(valid)])
        steps = (j - i) % n
        ls = "-" if steps == 1 else (0, (4, 3))
        a_seg = [angles[i], angles[j] if i < j else angles[j] + 2 * np.pi]
        ax.plot(a_seg, [vals[i], vals[j]], color=color, linestyle=ls,
                linewidth=2.2, zorder=2, solid_capstyle="round")

    ax.scatter(angles[valid], vals[valid], color=color, marker=marker,
               s=34, zorder=3, edgecolor="white", linewidth=0.6)

    ax.plot([], [], color=color, marker=marker, markersize=6,
            linewidth=2.2, label=model_display[model])

ax.set_xticks(angles)
ax.set_xticklabels(display_labels)
ax.tick_params(axis="x", pad=10)

# Side-align labels so they don't run into the chart.
for label, angle_rad in zip(ax.get_xticklabels(), angles):
    deg = np.degrees(angle_rad)
    if np.isclose(deg, 0) or np.isclose(deg, 180):
        label.set_horizontalalignment("center")
    elif 0 < deg < 180:
        label.set_horizontalalignment("left")
    else:
        label.set_horizontalalignment("right")

# Move the radial tick labels into the empty top-left wedge.
ax.set_rlabel_position(-360 / (2 * n))
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=9, color="0.4")
ax.set_ylim(0, 1.03)
ax.grid(True, color="0.85", linewidth=0.7)
ax.spines['polar'].set_visible(False)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=min(len(models), 2),
    frameon=False,
    handletextpad=0.5,
    columnspacing=1.5,
)

if recovery_missing.any():
    missing_display = ", ".join(
        model_display[m] for m in recovery_missing.index[recovery_missing])
    fig.text(
        0.5, 0.1,
        f"* dashed segment: recovery rate undefined (no iteration-1 failures) for {missing_display}",
        ha="center", va="bottom", fontsize=9, style="italic", color="0.3",
    )

fig.tight_layout(pad=0)
fig.savefig(OUT / "model_characteristics.pdf", pad_inches=0)
plt.show()

metrics.rename(index=model_display).round(3)